# GraphFlow (Workflows)

In this section We’ll learn how to create an multi-agent workflow using `GraphFlow`, or simply “flow” for short. It uses structured execution and precisely controls how agents interact to accomplish a task.<br><br>
We’ll first show you how to create and run a flow. We’ll then explain how to observe and debug flow behavior, and discuss important operations for managing execution.<br><br>
AutoGen AgentChat provides a team for directed graph execution:<br>
- **GraphFlow**: A team that follows a `DiGraph` to control the execution flow between agents. Supports sequential, parallel, conditional, and looping behaviors.

***Reference URL:-***
https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/graph-flow.html#sequential-flow

### GraphFlow

In [23]:
import asyncio
from autogen_agentchat.agents import AssistantAgent,UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import TextMentionTermination

In [22]:
# Load API key

import os
from dotenv import load_dotenv

load_dotenv()
api_key= os.getenv('OPENAI_API_KEY')

In [9]:
# Model client

model_client= OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

model_client

### DigraphBuilder GraphFlow

In [3]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

### Sequential Flow
We will begin by creating a simple workflow where a writer drafts a paragraph and a reviewer provides feedback. This graph terminates after the reviewer comments on the writer. <br>
Note, the flow automatically computes all the source and leaf nodes of the graph and the execution starts at all the source nodes in the graph and completes execution when no nodes are left to execute.

In [10]:
writer= AssistantAgent(
    name="Writer",
    description="A writer agent that generates text based on user input.",
    model_client=model_client,
    system_message="You are a creative writer. Please write a story based on the user's input.",
)

reviewer= AssistantAgent(
    name="Reviewer",
    description="A reviewer agent that provides feedback on the text generated by the writer.",
    model_client=model_client,
    system_message="You are a reviewer. Please provide feedback on the text generated by the writer.",
)

In [11]:
# Build the Graph

builder= DiGraphBuilder()
builder.add_node(writer).add_node(reviewer)
builder.add_edge(writer, reviewer)

graph= builder.build()

In [12]:
# Visualize The Graph

graph

DiGraph(nodes={'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)

In [ ]:
DiGraph(nodes={
    
    'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 
    
    'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')
    }, 
    default_start_node=None)

In [13]:
# Define Teams

team= GraphFlow([writer,reviewer], graph)
team

In [14]:
stream= team.run_stream(task="Write a good poem about India in less than 30 words.")

async for event in stream:
    print(event)

id='2558db42-b415-4a80-88b1-2a38acfc5e8d' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 34, 63388, tzinfo=datetime.timezone.utc) content='Write a good poem about India in less than 30 words.' type='TextMessage'
id='b9ca8079-011e-4572-a130-3d616963e387' source='Writer' models_usage=RequestUsage(prompt_tokens=41, completion_tokens=41) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 36, 327613, tzinfo=datetime.timezone.utc) content='Land of colors, diverse and grand,  \nRivers caress the fertile sand.  \nMountains whisper ancient tales,  \nIndia, where unity prevails.  \nIn vibrant dance, hearts expand.  ' type='TextMessage'
id='83283b4a-5d03-4177-864c-942d205d5484' source='Reviewer' models_usage=RequestUsage(prompt_tokens=88, completion_tokens=130) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 857587, tzinfo=datetime.timezone.utc) content='This is a beautifully succinct poem that captures the essence of India 

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 34, 63388, tzinfo=datetime.timezone.utc) 
content='Write a good poem about India in less than 30 words.' type='TextMessage'

source='Writer' models_usage=RequestUsage(prompt_tokens=41, completion_tokens=41) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 36, 327613, tzinfo=datetime.timezone.utc) 
content='Land of colors, diverse and grand,  \nRivers caress the fertile sand.  \nMountains whisper ancient tales,  \nIndia, where unity prevails.  \nIn vibrant dance, hearts expand.  ' type='TextMessage'

source='Reviewer' models_usage=RequestUsage(prompt_tokens=88, completion_tokens=130) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 857587, tzinfo=datetime.timezone.utc) 
content='This is a beautifully succinct poem that captures the essence of India in just a few lines. The imagery of diverse landscapes and cultural richness is effectively conveyed. The line "Mountains whisper ancient tales" brings a mystical touch, while "unity prevails" highlights India\'s core value of unity in diversity. The poem maintains a rhythm that complements its theme. Overall, it\'s a well-crafted piece that evokes both visual and emotional resonance. \n\nFor improvement, you might consider adding a specific reference to a cultural aspect, like a festival or a landmark, to further anchor the reader in India\'s unique identity. But for the constraints given, you have done an excellent job.' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 859591, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='2558db42-b415-4a80-88b1-2a38acfc5e8d', 
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 34, 63388, tzinfo=datetime.timezone.utc), 
                      content='Write a good poem about India in less than 30 words.', type='TextMessage'), TextMessage(id='b9ca8079-011e-4572-a130-3d616963e387', 
                                                                                                                       
                      source='Writer', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=41), metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 36, 327613, tzinfo=datetime.timezone.utc), 
                      content='Land of colors, diverse and grand,  \nRivers caress the fertile sand.  \nMountains whisper ancient tales,  \nIndia, where unity prevails.  \nIn vibrant dance, hearts expand.  ', type='TextMessage'), TextMessage(id='83283b4a-5d03-4177-864c-942d205d5484', 
                                                                                                                                                                                                                                                  
                      source='Reviewer', models_usage=RequestUsage(prompt_tokens=88, completion_tokens=130), metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 857587, tzinfo=datetime.timezone.utc), 
                      content='This is a beautifully succinct poem that captures the essence of India in just a few lines. The imagery of diverse landscapes and cultural richness is effectively conveyed. The line "Mountains whisper ancient tales" brings a mystical touch, while "unity prevails" highlights India\'s core value of unity in diversity. The poem maintains a rhythm that complements its theme. Overall, it\'s a well-crafted piece that evokes both visual and emotional resonance. \n\nFor improvement, you might consider adding a specific reference to a cultural aspect, like a festival or a landmark, to further anchor the reader in India\'s unique identity. But for the constraints given, you have done an excellent job.', type='TextMessage'), 
                      
                      StopMessage(id='e4594e5e-6988-4349-843a-ada7802eedd9', source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 859591, tzinfo=datetime.timezone.utc), content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received'